# Paper 26 neck on YOLO26n: LSA before the top-down fusion (backbone = proposed ADown + ECA)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Proposed (fixed for the whole neck study):** ADown at layers 3, 5, 7 + ECA after layers 6 (k = 3) and 8 (k = 5) |
| **Neck (layers 11–22)** | **Paper 26 (SLR-YOLO):** **LSA** (Lesion Selection Attention, LSKA-style) on the backbone feature that enters each top-down fusion node (layer 12: backbone P4, layer 15: backbone P3), then the usual concat |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

**Not taken from Paper 26 here:** DSE (layer 8, +53% GFLOPs, rejected in the backbone study) and MSRA (C2PSA, a backbone module, tested in the backbone study).

The reference row of the neck table is the **proposed backbone with the YOLO26n neck** (`final_backbone.ipynb`); it is not trained here.
Same training settings as every other row (100 epochs, seed 0, MuSGD, imgsz 640, batch 16).

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**. Then **Run All**.

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)

import time
NOTEBOOK_START = time.time()  # used by the Kaggle 12 h guard in the training cell


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Backbone (fixed): ADown (3, 5, 7) + ECA (6, 8)

The same backbone as `final_backbone.ipynb`. ADown is set in the YAML; ECA is added by the trainer after `yolo26n.pt` is loaded.

In [ ]:
%%writefile yolo26n-final-backbone.yaml
# YOLO26n with the proposed backbone: stride-2 Conv at layers 3, 5, 7 -> ADown (ECA after 6, 8 is added by the trainer).
# Neck and head = YOLO26n base.
nc: 80 # overridden by the dataset
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, ADown, [256]] # 3-P3/8   (Conv -> ADown)
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, ADown, [512]] # 5-P4/16  (Conv -> ADown)
  - [-1, 2, C3k2, [512, True]] # 6  (+ ECA k=3)
  - [-1, 1, ADown, [1024]] # 7-P5/32 (Conv -> ADown)
  - [-1, 2, C3k2, [1024, True]] # 8  (+ ECA k=5)
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

In [ ]:
%%writefile eca_backbone.py
"""Proposed backbone for YOLO26n: ECA after the C3k2 at layer 6 (P4, k=3) and layer 8 (P5, k=5). ADown (3, 5, 7) is in the YAML."""
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C3k2
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

ECA_LAYERS = {6: 3, 8: 5}  # layer index: ECA kernel size


class ECA(nn.Module):
    """Efficient Channel Attention: global avg pool -> conv across channels (kernel k) -> sigmoid -> scale. 1-D conv as 4-D Conv2d."""

    def __init__(self, k=3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(1, 1, (k, 1), padding=(k // 2, 0), bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.pool(x).transpose(1, 2)  # (B, C, 1, 1) -> (B, 1, C, 1)
        y = self.act(self.conv(y)).transpose(1, 2)  # back to (B, C, 1, 1)
        return x * y


class C3k2ECA(nn.Module):
    """A pretrained C3k2 block followed by ECA."""

    def __init__(self, block, k=3):
        super().__init__()
        self.block = block
        self.eca = ECA(k)

    def forward(self, x):
        return self.eca(self.block(x))


def add_eca(model, layers=ECA_LAYERS):
    """Wrap the C3k2 at the given layers with ECA (the C3k2 keeps its weights). Returns the model."""
    for i, k in layers.items():
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        new = C3k2ECA(old, k).to(next(old.parameters()).device)
        new.i, new.f, new.type = old.i, old.f, "C3k2ECA"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


class ECATrainer(DetectionTrainer):
    """Builds the model from the ADown YAML, loads the pretrained weights, then adds ECA after the C3k2 at layers 6 and 8."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_eca(model)
        if RANK in {-1, 0}:
            print(f"Proposed backbone: ADown at layers 3, 5, 7 + ECA after layers {list(ECA_LAYERS)} (k = {list(ECA_LAYERS.values())})")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

In [ ]:
%%writefile fix_eca_ncnn.py
"""Re-export ECA-style models to NCNN without the Segmentation fault (papers 10, 17, 19, 49).

Why: ECA blocks do pool -> transpose(1,2) -> Conv2d(1,1,(k,1)) -> transpose. PNNX turns the first transpose into a
no-op Permute, so the 1-channel conv receives C channels and NCNN crashes (Segmentation fault).

Fix: the cross-channel 1-D conv is rewritten as an exactly equivalent 1x1 Conv2d(C, C) whose weight is a banded
matrix built from the trained kernel. Same math, same outputs, no transposes -> NCNN-safe.

Run inside the unzipped bench folder (needs bb_*.pt and the *_backbone.py file):
    python fix_eca_ncnn.py            # check + export every bb_*.pt that contains ECA (replaces <name>_ncnn_model)
    python fix_eca_ncnn.py --check    # only check equivalence, no export
Models without ECA blocks are left untouched.
"""
import glob
import importlib
import shutil
import sys
from pathlib import Path

import torch
import torch.nn as nn

sys.path.insert(0, ".")
for f in glob.glob("*_backbone.py"):  # torch needs these to unpickle the custom modules
    importlib.import_module(Path(f).stem)

from ultralytics import YOLO  # noqa: E402


class ECAConv1x1(nn.Module):
    """ECA with the cross-channel conv as a banded 1x1 Conv2d(C, C). scale=True -> x * att, False -> att only."""

    def __init__(self, conv1d: nn.Conv2d, c: int, scale: bool):
        super().__init__()
        w = conv1d.weight.detach().view(-1).float()  # (k,)
        k, p = w.numel(), conv1d.padding[0]
        band = torch.zeros(c, c)
        for o in range(c):  # out[o] = sum_j w[j] * in[o + j - p]  (zero padding)
            for j in range(k):
                i = o + j - p
                if 0 <= i < c:
                    band[o, i] = w[j]
        self.scale = scale
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(c, c, 1, bias=False)
        self.conv.weight.data.copy_(band.view(c, c, 1, 1))
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.act(self.conv(self.pool(x)))
        return x * y if self.scale else y


def is_eca(m):
    """A block whose .conv is Conv2d(1, 1, (k, 1)) with k > 1 - the transpose-based ECA pattern."""
    c = getattr(m, "conv", None)
    return (isinstance(c, nn.Conv2d) and c.in_channels == 1 and c.out_channels == 1
            and c.kernel_size[1] == 1 and c.kernel_size[0] > 1 and not isinstance(m, ECAConv1x1))


@torch.no_grad()
def patch(net):
    """Replace every ECA block. Returns (count, max abs difference old vs new)."""
    names = [n for n, m in net.named_modules() if is_eca(m)]
    if not names:
        return 0, 0.0
    chans = {}

    def hook(name):
        def f(m, inp, out):  # must return None, otherwise it would replace the block's output
            chans.setdefault(name, inp[0].shape[1])
        return f

    hooks = [net.get_submodule(n).register_forward_hook(hook(n)) for n in names]
    net(torch.zeros(1, 3, 640, 640))
    for h in hooks:
        h.remove()

    diff = 0.0
    for n in names:
        old, c = net.get_submodule(n), chans[n]
        x = torch.randn(2, c, 20, 20)
        ref = old(x)
        new = None
        for scale in (True, False):  # find which variant this block is (returns x*att or att only)
            cand = ECAConv1x1(old.conv, c, scale).eval()
            out = cand(x)
            if out.shape == ref.shape and (out - ref).abs().max().item() < 1e-4:
                new = cand
                diff = max(diff, (out - ref).abs().max().item())
                break
        if new is None:
            sys.exit(f"Block {n} ({type(old).__name__}) did not match - not exporting.")
        parent, _, attr = n.rpartition(".")
        setattr(net.get_submodule(parent) if parent else net, attr, new)
    return len(names), diff


def main():
    pts = sorted(p for p in glob.glob("bb_*.pt") if not p.endswith("_fixed.pt"))
    if not pts:
        sys.exit("No bb_*.pt found. Run this inside the unzipped bench folder.")
    for pt in pts:
        yolo = YOLO(pt, task="detect")
        net = yolo.model.float().eval()
        n, diff = patch(net)
        if n == 0:
            print(f"{pt}: no ECA blocks - left as is.")
            continue
        print(f"{pt}: replaced {n} ECA blocks, max block output difference = {diff:.2e}  -> Check OK")
        if "--check" in sys.argv:
            continue

        name = Path(pt).stem
        tmp = Path(f"{name}_fixed.pt")
        yolo.save(tmp)
        exported = Path(YOLO(str(tmp), task="detect").export(format="ncnn", imgsz=640))
        out_dir = Path(f"{name}_ncnn_model")
        if out_dir.exists():
            shutil.rmtree(out_dir)
        shutil.move(str(exported), out_dir)
        tmp.unlink(missing_ok=True)
        Path(f"{name}_fixed.torchscript").unlink(missing_ok=True)
        print(f"{pt}: Done -> {out_dir}/")
    print("All done. Now run: python pi5_bench.py")


if __name__ == "__main__":
    main()

## 4. Paper 26 neck: LSA (Lesion Selection Attention)

LSA cleans the backbone feature $X$ before it is fused with the upsampled deeper feature (Paper 26, Section 3.3, Eq. 6–9):

$$Z = \mathrm{Conv}_{1\times1}(X), \quad Z_s = \mathrm{DW}_{5\times5}(Z), \quad Z_g = \mathrm{DW}_{7\times7,\,d=3}(Z_s), \quad Z_c = Z_s + Z_g$$
$$\bar Z_c = Z_c \odot \sigma\big(W_2\,\delta(W_1\,\mathrm{GAP}(Z_c))\big) \quad\text{(channel selection)}$$
$$A = \sigma\big(\mathrm{Conv}_{7\times7}[\mathrm{Mean}_c(\bar Z_c),\ \mathrm{Max}_c(\bar Z_c)]\big), \qquad X_\text{LSA} = X \odot A + X, \qquad F = \mathrm{Concat}(\mathrm{Up}(F_{l+1}),\ X_\text{LSA})$$

The local 5 × 5 depthwise conv and the dilated 7 × 7 depthwise conv (dilation 3, receptive field 23 × 23) are the LSKA large-kernel idea;
$A$ is one spatial mask shared by all channels, and the residual keeps the original feature.

**Where:**

| Layer | YOLO26n neck | This notebook |
|---|---|---|
| 12 | Concat(up P5, backbone P4) | Concat(up P5, **LSA(backbone P4)**) |
| 15 | Concat(up P4, backbone P3) | Concat(up P4, **LSA(backbone P3)**) |
| 18, 21 | Concat (bottom-up) | unchanged |

Paper 26 applies LSA before the top-down fusion (Eq. 9 fuses $X_\text{LSA}$ with $\mathrm{Up}(F_{l+1})$), so only the two top-down nodes change.
Channel counts stay the same, so every C3k2 keeps its pretrained weights.

**Pretrained start.** The 7 × 7 mask conv starts with zero weights and bias −3, so $A \approx 0.05$ and $X_\text{LSA} \approx 1.05\,X$: the pretrained neck
sees almost the plain feature at the start (Paper 26 does not describe its initialisation).

**Assumptions** (not given exactly in Paper 26): the 1 × 1 compression halves the channels, kernel sizes 5 and 7 with dilation 3 (the LSKA defaults),
channel-gate reduction 16, mask conv 7 × 7.

**Export.** Depthwise and dilated depthwise convs, mean/max over channels, sigmoid, multiply, add and concat, all supported by NCNN.

In [ ]:
%%writefile lsa_neck.py
"""Paper 26 neck for YOLO26n. Backbone = proposed ADown (3, 5, 7) + ECA (6, 8). Head = YOLO26n."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Concat, Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from eca_backbone import ECA_LAYERS, add_eca

LSA_LAYERS = (12, 15)  # top-down nodes: LSA on the backbone input (layer 6 at node 12, layer 4 at node 15)
MASK_BIAS = -3.0  # sigmoid(-3) = 0.05: LSA starts close to the identity


class LSA(nn.Module):
    """Lesion Selection Attention (Paper 26, Eq. 6-9): X * A + X."""

    def __init__(self, c, ks=5, kg=7, dg=3, r=16, k=7):
        super().__init__()
        c_ = max(c // 2, 8)
        self.cv = Conv(c, c_, 1)  # phi_1x1: compression
        self.dw_s = nn.Conv2d(c_, c_, ks, 1, ks // 2, groups=c_)  # local lesion evidence
        self.dw_g = nn.Conv2d(c_, c_, kg, 1, (kg // 2) * dg, dilation=dg, groups=c_)  # wide context
        h = max(c_ // r, 8)
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c_, h, 1), nn.ReLU(inplace=True), nn.Conv2d(h, c_, 1), nn.Sigmoid())
        self.mask = nn.Conv2d(2, 1, k, 1, k // 2)
        nn.init.zeros_(self.mask.weight)
        nn.init.constant_(self.mask.bias, MASK_BIAS)

    def forward(self, x):
        zs = self.dw_s(self.cv(x))
        zc = zs + self.dw_g(zs)
        zc = zc * self.gate(zc)  # channel selection
        a = torch.sigmoid(self.mask(torch.cat((zc.mean(1, keepdim=True), zc.amax(1, keepdim=True)), 1)))
        return x * a + x


class LSAFusion(nn.Module):
    """Concat(up deeper feature, LSA(backbone feature))."""

    def __init__(self, c_lat):
        super().__init__()
        self.lsa = LSA(c_lat)

    def forward(self, xs):
        return torch.cat((xs[0], self.lsa(xs[1])), 1)


@torch.no_grad()
def _read_channels(model, layers, pick):
    """One small forward pass to read a channel count at each layer; pick(inp, out) returns the tensor to measure."""
    chans, hooks = {}, []

    def record(i):
        def hook(m, inp, out):  # must return None, otherwise it would replace the layer output
            chans.setdefault(i, pick(inp, out).shape[1])
        return hook

    for i in layers:
        hooks.append(model.model[i].register_forward_hook(record(i)))
    was_training = model.training
    model.eval()
    model(torch.zeros(1, 3, 64, 64, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return chans


def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(model.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_lsa(model, layers=LSA_LAYERS):
    """Replace the top-down Concat at the given layers with Concat(up, LSA(backbone))."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Concat) and len(old.f) == 2 and old.f[0] == -1, f"layer {i} is not a 2-input Concat"
    chans = _read_channels(model, layers, lambda inp, out: inp[0][1])
    for i in layers:
        _swap(model, i, LSAFusion(chans[i]), "LSAFusion")
    return model


class LSANeckTrainer(DetectionTrainer):
    """Builds the ADown YAML, loads the pretrained weights, adds ECA (layers 6, 8), then changes the neck: LSA (Paper 26) before the top-down fusion at layers 12, 15."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_eca(model)
        add_lsa(model)
        if RANK in {-1, 0}:
            print(f"Backbone: ADown 3, 5, 7 + ECA after {list(ECA_LAYERS)} | Neck: LSA (Paper 26) before the top-down fusion at layers 12, 15")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


In [ ]:
import importlib, eca_backbone, lsa_neck
importlib.reload(eca_backbone); importlib.reload(lsa_neck)
from eca_backbone import add_eca
from lsa_neck import add_lsa, LSANeckTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

MODEL_YAML = str(WORK / "yolo26n-final-backbone.yaml")

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs proposed backbone vs proposed backbone + Paper 26 neck
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
prop = add_eca(DetectionModel(yaml_model_load(MODEL_YAML), nc=len(names), verbose=False)).eval()
neck = add_lsa(deepcopy(prop)).eval()
with torch.no_grad():
    _ = neck(torch.zeros(1, 3, 640, 640))  # full forward check
    _ = neck(torch.zeros(1, 3, 640, 352))  # rect-style input, as in validation
    neck.train(); _ = neck(torch.zeros(2, 3, 640, 640)); neck.eval()  # training path too
print("neck layers (12, 15, 18, 21):", [type(neck.model[i]).__name__ for i in (12, 15, 18, 21)])
for tag, m in (("base YOLO26n", base), ("Proposed backbone", prop), ("+ Paper 26 neck", neck)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.3f} M | GFLOPs {gf:.2f}")
print("expected: + LSA neck slightly above the proposed backbone (~2.03 M | ~4.49 GFLOPs)")

## 5. Training

Only the proposed backbone + Paper 26 neck model is trained (about 2–3 h on a T4).

**Kaggle 12 h limit:** if the run gets close to 12 hours, training stops by itself after the epoch that passes 10.5 h (counted from the setup cell) and the notebook continues with the evaluation, export and zip. The cell prints how many epochs were trained; write that number in the paper if it is below 100.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("Proposed Backbone + LSA Neck (Paper 26)", "nk_lsa"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

# Kaggle stops a session after 12 hours. Training stops after the epoch that passes TRAIN_STOP_HOURS (counted from the
# setup cell), so the test evaluation, NCNN export and zip still finish in time. best.pt is the best epoch reached so far.
TRAIN_STOP_HOURS = 10.5

def stop_before_kaggle_limit(trainer):
    hours = (time.time() - NOTEBOOK_START) / 3600
    if hours > TRAIN_STOP_HOURS and not trainer.stop:
        trainer.stop = True
        print(f"\n⏱ {hours:.1f} h since the notebook started: training stops after epoch {trainer.epoch + 1} (Kaggle 12 h limit).")

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO(MODEL_YAML)
    model.add_callback("on_fit_epoch_end", stop_before_kaggle_limit)
    model.train(trainer=LSANeckTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    n_ep = len(pd.read_csv(Path(PROJECT) / f"{run}_s{SEED}" / "results.csv"))
    print(f"{run}: trained {n_ep} of {EPOCHS} epochs" + ("" if n_ep >= EPOCHS else " (stopped early: patience or the 12 h guard)"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 6. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as every other row.

In [ ]:
import eca_backbone, lsa_neck  # must be imported before loading the checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 7. Package for Raspberry Pi 5 and Jetson

Creates `paper26_neck_edge_bench/` with:

| Item | Used on |
|---|---|
| `nk_lsa_ncnn_model/` (NCNN model, imgsz 640, ECA as 1×1 conv) | Raspberry Pi 5 |
| `nk_lsa.pt` + `eca_backbone.py` + `lsa_neck.py` | Jetson |
| `bench_images/` (the same 50 real test images, seed 0) | both |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Before the NCNN export, the 2 ECA blocks of the backbone are rewritten as exactly equivalent banded 1×1 convs (`fix_eca_ncnn.py`),
because the transpose-based ECA crashes NCNN. The `.pt` file for the Jetson keeps the original ECA.

In [ ]:
import eca_backbone, lsa_neck  # needed to load the checkpoint
from fix_eca_ncnn import patch  # ECA -> banded 1x1 conv, NCNN-safe

EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper26_neck_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("eca_backbone.py", EDGE / "eca_backbone.py")
shutil.copy("lsa_neck.py", EDGE / "lsa_neck.py")

# the same 50 real test images as the earlier packages (same seed and selection)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    # NCNN-safe copy: ECA -> equivalent banded 1x1 conv (checked block by block), then export
    yolo = YOLO(str(w), task="detect")
    n, diff = patch(yolo.model.float().eval())
    print(f"{run}: replaced {n} ECA blocks for NCNN, max block output difference = {diff:.2e}")
    assert n == 2, "expected 2 ECA blocks (layers 6 and 8)"
    tmp = WORK / f"{run}_ncnn_safe.pt"
    yolo.save(tmp)
    p = YOLO(str(tmp), task="detect").export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import eca_backbone, lsa_neck  # needed to load the checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
print("\nEdge package ready. Run section 9 to zip everything.")

### How to run on the Raspberry Pi 5

Copy `paper26_neck_all_results.zip` to the Pi and run the bench script:
```bash
~/fps/bench_paper.sh paper26_neck_all_results.zip
```

## 8. Neck table row

The reference row is the **proposed backbone with the YOLO26n neck** from `final_backbone.ipynb`. Type its values once they are known
(and the FPS of this model from the Pi 5), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 6 first in the same session (it fills `METRICS`).

In [ ]:
# reference row: proposed backbone + YOLO26n neck (fill in from final_backbone.ipynb)
BASE_ROW = {"Variant": "Proposed Backbone + YOLO26n Neck", "Params (M)": 2.03, "GFLOPs": 4.49,
            "Precision": None, "Recall": None, "F1-score": None, "mAP@50": None,
            "FPS (Rasp-5)": None, "FPS (Jetson)": None}

FPS_PI5    = {"nk_lsa": None}
FPS_JETSON = {"nk_lsa": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 9. Zip all results

Run this last. It creates **`paper26_neck_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper26_neck_edge_bench/` | NCNN + `.pt` model, module files, 50 bench images, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `nk_lsa_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |
| `yolo26n-final-backbone.yaml`, `eca_backbone.py`, `lsa_neck.py` | the model definition |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper26_neck_all_results.zip"
folders = [WORK / "paper26_neck_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")
    if Path(MODEL_YAML).exists():
        z.write(MODEL_YAML, Path(MODEL_YAML).name)
    z.write("eca_backbone.py", "eca_backbone.py")
    z.write("lsa_neck.py", "lsa_neck.py")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")